# 03. Preparación, pipeline y modelos preliminares

**Responsable:** José · **Secciones de la guía:** 4.2.4, 4.2.5, 4.2.6, 4.2.7, 4.2.8, 4.2.9

Aplica las reglas de calidad y limpieza sobre el dataset crudo, realiza la separación temporal train/test sin fuga de datos, define los baselines de referencia (Dummy y Regla de Mercado) y entrena los modelos predictivos preliminares.

> Antes de subir cambios: *Kernel > Restart & Run All*.


## Carga de datos

Cargamos los datos sin procesar usando `cargar_datos()`.


In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.dummy import DummyRegressor

from src import config
from src import limpieza
from src.datos import cargar_datos
from src.baseline import ReglaMercadoM2
from src.metricas import evaluar, errores_porcentuales, tabla_comparativa

pd.set_option("display.max_columns", 30)
plt.rcParams["figure.dpi"] = 110

df_raw = cargar_datos()
print(f"Dataset cargado: {len(df_raw):,} registros")


Dataset cargado: 119,980 registros


In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 119980 entries, 0 to 119979
Data columns (total 16 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   periodo            119980 non-null  str    
 1   anio               119980 non-null  int64  
 2   trimestre          119980 non-null  int64  
 3   precio_usd         119980 non-null  float64
 4   tipo_cambio        119980 non-null  float64
 5   ipc                119980 non-null  float64
 6   precio_soles       119980 non-null  float64
 7   precio_soles_2009  119980 non-null  float64
 8   distrito           119980 non-null  str    
 9   superficie_m2      119980 non-null  float64
 10  habitaciones       119967 non-null  float64
 11  banos              119944 non-null  float64
 12  garajes            118429 non-null  float64
 13  piso               115562 non-null  float64
 14  vista_exterior     115923 non-null  float64
 15  antiguedad_anios   118258 non-null  float64
dtypes: float64(12

## 1. Calidad y preparación de datos (4.2.4)

Aplicamos las reglas de limpieza formuladas a partir de las limitaciones identificadas en el notebook 01 y el EDA (02):
1. Deduplicación de anuncios idénticos antes de particionar.
2. Normalización de cadenas de texto en distritos (unificación de mayúsculas/espacios).
3. Corrección de valores físicamente imposibles (antigüedad > 150 años, baños con decimales inválidos).
4. Descarte de columnas redundantes y con fuga directa de datos (`precio_soles`, `precio_soles_2009`, `tipo_cambio`, `ipc`, `periodo`).
5. Descarte de variables no continuadas en la recolección (`piso`, `vista_exterior`).


In [2]:
# 1. Deduplicación
df_dedup = limpieza.eliminar_duplicados(df_raw)
n_dup = len(df_raw) - len(df_dedup)

# 2. Normalización de distritos
df_norm = limpieza.normalizar_distritos(df_dedup)

# 3. Corrección de valores imposibles
df_corr = limpieza.corregir_valores_imposibles(df_norm)

# 4. Descarte de columnas con fuga y redundantes
cols_a_descartar = config.COLUMNAS_FUGA + config.COLUMNAS_REDUNDANTES + ["piso", "vista_exterior"]
df_clean = df_corr.drop(columns=[c for c in cols_a_descartar if c in df_corr.columns])

print(f"Filas duplicadas removidas: {n_dup:,}")
print(f"Dimensiones tras limpieza inicial: {df_clean.shape}")
print(f"Columnas activas: {list(df_clean.columns)}")


Filas duplicadas removidas: 2,730
Dimensiones tras limpieza inicial: (117250, 9)
Columnas activas: ['anio', 'trimestre', 'precio_usd', 'distrito', 'superficie_m2', 'habitaciones', 'banos', 'garajes', 'antiguedad_anios']


## 2. Separación train / test (4.2.5)

**Estrategia:** Partición temporal estricta (*Hold-out out-of-time*). Los datos se ordenan cronológicamente y se dividen en:
- **Entrenamiento (Train):** Anuncios hasta el año 2023 (`config.ANIO_FIN_TRAIN`).
- **Evaluación (Test):** Anuncios de 2024 en adelante (`config.ANIO_INICIO_TEST`).

**Justificación:** En el mercado inmobiliario, el precio depende del ciclo económico e inflación. Una separación aleatoria provocaría fuga temporal (*data leakage*), pues el modelo memorizaría el nivel de precios de un trimestre y predeciría departamentos del mismo trimestre en test. La partición temporal simula exactamente el uso real: predecir precios futuros con datos históricos.

**Filtro de precios extremos:** El filtro `FiltroPrecioM2Extremo` (regla 9) se ajusta **únicamente sobre el conjunto de entrenamiento** para evitar fuga de información hacia el conjunto de test.


In [3]:
train_raw = df_clean[df_clean["anio"] <= config.ANIO_FIN_TRAIN].copy()
test_raw = df_clean[df_clean["anio"] >= config.ANIO_INICIO_TEST].copy()

# Ajuste del filtro de outliers solo sobre train
filtro_m2 = limpieza.FiltroPrecioM2Extremo().fit(train_raw)
train = train_raw[~filtro_m2.es_extremo(train_raw)].reset_index(drop=True)
test = test_raw[~filtro_m2.es_extremo(test_raw)].reset_index(drop=True)

print(f"Train: {len(train):,} anuncios ({train['anio'].min()}–{train['anio'].max()})")
print(f"Test:  {len(test):,} anuncios ({test['anio'].min()}–{test['anio'].max()})")
print(f"Proporción Train / Test: {len(train)/(len(train)+len(test)):.1%} / {len(test)/(len(train)+len(test)):.1%}")


Train: 92,009 anuncios (1998–2023)
Test:  25,161 anuncios (2024–2026)
Proporción Train / Test: 78.5% / 21.5%


## 3. Baselines de referencia (4.2.7)

Evaluamos los modelos de referencia que sirven de umbral mínimo de utilidad:
1. **Dummy Regressor (Media):** Predice siempre la media de precios de train.
2. **Dummy Regressor (Mediana):** Predice siempre la mediana de precios de train.
3. **Regla de Mercado (Distrito × m²):** Aplica la mediana de US$/m² reciente del distrito multiplicada por la superficie del departamento.


In [4]:
X_train, y_train = train.drop(columns=[config.OBJETIVO]), train[config.OBJETIVO]
X_test, y_test = test.drop(columns=[config.OBJETIVO]), test[config.OBJETIVO]

baselines = {
    "Dummy (Media)": DummyRegressor(strategy="mean").fit(X_train, y_train),
    "Dummy (Mediana)": DummyRegressor(strategy="median").fit(X_train, y_train),
    "Regla de mercado (distrito × m²)": ReglaMercadoM2(anios_referencia=1).fit(X_train, y_train),
}

resultados_baselines = {}
predicciones_baselines = {}
for nombre, modelo in baselines.items():
    y_pred = modelo.predict(X_test)
    predicciones_baselines[nombre] = y_pred
    resultados_baselines[nombre] = evaluar(y_test, y_pred)

tabla_comparativa(resultados_baselines)


,MAE (US$),RMSE (US$),Error % mediano,Error % medio (MAPE),Anuncios con error <= 10%
Dummy (Media),"53,868","70,918",27.7%,36.6%,17.3%
Dummy (Mediana),"59,903","81,544",31.6%,34.3%,17.0%
Regla de mercado (distrito × m²),"29,183","40,003",15.0%,18.6%,34.3%


## 4. Pipeline de preprocesamiento (4.2.6)

Pendiente: Implementar `ColumnTransformer` formal de scikit-learn con imputación justificada y codificación de variables categóricas.


In [5]:
# Pendiente: Definición del preprocesador formal


## 5. Modelos preliminares (4.2.8)

Pendiente: Entrenamiento de modelos supervisados (ej. Ridge Regressor y Gradient Boosting / Random Forest) superando la vara del 15% de la regla de mercado.


In [6]:
# Pendiente: Modelos supervisados


## 6. Evaluación y diagnóstico de errores (4.2.9)

Pendiente: Comparación de métricas de modelos supervisados frente a la Regla de Mercado y análisis de residuales por distrito y segmento.


In [7]:
# Pendiente: Evaluación final
